In [1]:

import json
import os
import sys
from dotenv import load_dotenv
from pathlib import Path

%load_ext autoreload
%autoreload 2


load_dotenv()
root_path = Path().absolute().parent.parent.parent
sys.path.append(str(root_path))


from functions.llm_config import LLMConfig
from functions.retrieval import QuestionRetriever
from functions.query_decomposer import QueryDecomposer
from functions.sqldatabase_langchain_utils import SQLDatabaseLangchainUtils
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage
import paths as paths

import re
from pydantic import BaseModel, Field
from typing import List



from eval_agent.text2sql_agent.text_to_sql.text_to_sql_extended_schema import TextToSQLExtendedSchema




In [2]:

O3 = LLMConfig(provider="azure").get_llm(model="o3-mini", reasoning_effort="high")
GPT4O = LLMConfig(provider="azure").get_llm(model="gpt-4o")



decomposer = QueryDecomposer(
    GPT4O,
    paths.PROMPT_DECOMPOSER_FILE,
    False
)

experiment = os.getenv("EXPERIMENT_NAME")


prompt_path = paths.EXTENDED_SCHEMA_PROMPT
DATASET_SYNTHETIC_PATH = paths.DATASET_SYNTHETIC_PATH
EMBEDDINGS_PATH = paths.EMBEDDINGS_PATH


with open(paths.DB_CONNECTION_FILE, "r") as f:
    db_connection = json.load(f)

if DATASET_SYNTHETIC_PATH == "" or EMBEDDINGS_PATH == "":
    print("Dataset path or embeddings path is not set. Please check the .env configuration.")
    retriever = None

else:   
    retriever = QuestionRetriever(
        dataset_path=DATASET_SYNTHETIC_PATH,
        vectors_path=EMBEDDINGS_PATH,
        # vectorize=True
    )
    retriever.remove_duplicates()

/home/tomas/PUC/TecGraf/conversational_bm/benchmark_nl_databases_conversational_interfaces/functions/query_decomposer.py:37: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  self.query_analyzer = LLMChain(llm=llm, prompt=self.prompt_builder, verbose=False)


17030
17030


In [3]:
sqldatabase = SQLDatabaseLangchainUtils(db_connection)

/home/tomas/PUC/TecGraf/conversational_bm/benchmark_nl_databases_conversational_interfaces/venv/lib/python3.10/site-packages/langchain_community/utilities/sql_database.py:134: SAWarning: Did not recognize type 'GEOCOORD' of column 'coordinates'
  self._metadata.reflect(
/home/tomas/PUC/TecGraf/conversational_bm/benchmark_nl_databases_conversational_interfaces/venv/lib/python3.10/site-packages/langchain_community/utilities/sql_database.py:134: SAWarning: Did not recognize type 'GEOCOORD' of column 'source'
  self._metadata.reflect(
/home/tomas/PUC/TecGraf/conversational_bm/benchmark_nl_databases_conversational_interfaces/venv/lib/python3.10/site-packages/langchain_community/utilities/sql_database.py:134: SAWarning: Did not recognize type 'GEOCOORD' of column 'estuary'
  self._metadata.reflect(


In [4]:
t =  TextToSQLExtendedSchema(GPT4O, decomposer, retriever, prompt_path, debug=False)


def test(question):
    print("Running Test")

    result = t.translate_text_to_sql(question)

    
    result.sql_query = result.sql_query.replace("```sql", "").replace("```", "")
    if not result.sql_query.strip().upper().startswith("SELECT"):
        result.sql_query = "SELECT " + result.sql_query
    if "DISTINCT" in result.sql_query:
        result.sql_query = result.sql_query.replace("DISTINCT", "")
    print("Result")
    sql = result.sql_query
    print(result.sql_query)
    print(result.schema_linking_tables)
    print(sqldatabase.run_in_database(sql))
    

In [5]:
from eval_agent.text2sql_agent.tool import convert_text_to_sql_and_execute

question = "Qual é a capital de cuba?"

print(convert_text_to_sql_and_execute(question))

17030
17030
Decomposition
['Qual é a capital de cuba?', 'Qual é a capital de Cuba?']
(17030,)
(17030,)
DFE
Question: islands name 'Cuba'
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name "Cuba"
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: What are the islands in the MONDIAL_ISLAND table with the name "Cuba"?
SELECT NAME, ISLANDS, AREA, ELEVATION, TYPE, COORDINATES, META_REPCOL
FROM MONDIAL_ISLAND
WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name 'Cuba'?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name "Cuba"?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name 'Cuba'
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name "Cuba"
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name "Cuba"?
SELECT NAME FROM MONDIAL_ISL

/home/tomas/PUC/TecGraf/conversational_bm/benchmark_nl_databases_conversational_interfaces/venv/lib/python3.10/site-packages/pandas/io/sql.py:761: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(


Decomposition
['Qual é a capital de cuba?', 'Qual é a capital de Cuba?']
(17030,)
(17030,)
DFE
Question: islands name 'Cuba'
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name "Cuba"
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: What are the islands in the MONDIAL_ISLAND table with the name "Cuba"?
SELECT NAME, ISLANDS, AREA, ELEVATION, TYPE, COORDINATES, META_REPCOL
FROM MONDIAL_ISLAND
WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name 'Cuba'?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name "Cuba"?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name 'Cuba'
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: islands name "Cuba"
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name "Cuba"?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'

Question: What are the islands in the MONDIAL_ISLAND table with the name "Cuba"?
SELECT NAME, ISLANDS, AREA, ELEVATION, TYPE, COORDINATES, META_REPCOL
FROM MONDIAL_ISLAND
WHERE LOWER(NAME) = 'cuba'

Question: Which islands in the database have the name 'Cuba'?
SELECT NAME FROM MONDIAL_ISLAND WHERE LOWER(NAME) = 'cuba'


{'input': 'Qual é a capital de cuba?', 'schema_linking': ['mondial_gpt.country'], 'answer':      CAPITAL
0  La Habana, 'sql': "SELECT capital FROM mondial_gpt.country WHERE LOWER(name) = 'cuba'"}
